In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle

In [2]:
from nmf_utils import bandpass_filter_traces, get_peaks_maxsegments, plot_traces_peaks, plot_shaded_histogram, plot_raster, plot_population_activity

In [3]:
def main(path_save, file_stem):
    """
    Filter calcium traces, detect activity peaks, compute statistics, and save results.

    This function:
    1. Loads previously extracted ΔF/F traces from a pickle file.
    2. Applies a bandpass filter to remove slow drift and high-frequency noise.
    3. Detects calcium activity peaks using a statistical thresholding approach.
    4. Computes peak features such as start time, maximum, time-to-peak, and amplitude.
    5. Generates and saves visualizations (trace plots, histograms, raster plots, population activity).
    6. Saves processed peak data and parameters into a new pickle bundle.

    Parameters
    ----------
    path_save : str
        Directory where the input trace file is located and outputs will be saved.
    file_stem : str
        Base filename used to locate input and name output files.

    Returns
    -------
    None
        Results are saved to disk as:
        - '{file_stem}_traces.png'
        - '{file_stem}_peak_stats.png'
        - '{file_stem}_raster_peak_max.png'
        - '{file_stem}_raster_peak_start.png'
        - '{file_stem}_population_activity_plot.png'
        - '{file_stem}_peaks.pkl'

    Outputs (saved in bundle)
    -------------------------
    peaks_start : np.ndarray
        Start indices of detected peaks for each ROI.
    peaks_max : np.ndarray
        Peak maxima indices for each ROI.
    time_to_peak : np.ndarray
        Time (in seconds) from peak start to peak maximum.
    peaks_amp : np.ndarray
        Peak amplitudes (ΔF/F values).
    raster : np.ndarray
        Binary raster representation of peak activity over time.
    parameters : dict
        Dictionary containing filtering and peak detection parameters.

    Notes
    -----
    - The last frame of each trace is removed due to occasional artifacts.
    - A bandpass filter is applied before peak detection to improve signal quality.
    - Peak detection uses a percentile-based baseline and standard deviation thresholding.
    - Thresholds are averaged across all NMF components in the recording.
    - Outputs provide both single-cell and population-level activity summaries.
    """
    #Parameters
    #bandpass_filter_traces parameters
    fs=10 # aquisition frequency in Hz
    low_freq_thresh=0.1  #low bandpass threshold
    high_freq_thresh=3  #high bandpass threshold
    order=3 # bandpass filter order

    # get_peaks_maxsegments parameters
    precentile_threshold=80 #percentile to determine inter-peak intervals 
    strict=3 # time of SD deviation from inter-peak baseline to start the peak
    dF_thresh=0 #threshold for peak finding, if 0, then inter-peak baseline + n x SD is used
    file_data =os.path.join(path_save, f"{file_stem}_traces.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    traces = bundle["traces"][:,:-1] #remove the last frame because it is black in some recordings
    n_traces=traces.shape[0]
    traces_filt=bandpass_filter_traces(traces, fs=fs, low=low_freq_thresh, high=high_freq_thresh, order=order)
    if n_traces==1:
        spike_thresh=traces_filt.std(axis=1)*5
    else:
        spike_thresh=traces_filt.std(axis=1).mean()*5

    # Extract peaks timestamps and peaks amplitudes 
    peaks, amp = get_peaks_maxsegments(traces_filt, precentile_threshold=precentile_threshold, strict=strict, dF_thresh=dF_thresh)

    #Plot traces and save figures
    fig = plot_traces_peaks (traces_filt, peaks, spike_thresh)
    plt.savefig(os.path.join(path_save, file_stem +"_traces.png"), dpi=300)
    plt.close()

    #Calculate time-to-peak and amplitude and save
    #Peaks onsets
    peaks_start = {
        roi: [p[0] for p in plist]
        for roi, plist in peaks.items()
    }
    #Peaks maxima
    peaks_max = {
        roi: [p[1] for p in plist]
        for roi, plist in peaks.items()
    }

    peaks_start = np.array(pd.DataFrame.from_dict(peaks_start, orient="index"))
    peaks_max = np.array(pd.DataFrame.from_dict(peaks_max, orient="index"))
    time_to_peak = (peaks_max - peaks_start)/fs # in seconds
    peaks_amp = np.array(pd.DataFrame.from_dict(amp, orient="index"))

    # Plot time-to-peak and amplitude histograms
    to_plot1 = time_to_peak
    to_plot2 = peaks_amp
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    plot_shaded_histogram(to_plot1, ax=axs[0], xlabel="Time-to-peak (s)", ylabel="Count", title="Time-to-peak histogram (mean ± SD across ROIs)")
    plot_shaded_histogram(to_plot2, ax=axs[1], xlabel="Amplitute (DF/F)", ylabel="Count", title="Peak amplitude histogram (mean ± SD across ROIs)")
    plt.savefig(os.path.join(path_save, file_stem+"_peak_stats.png"), dpi=300)
    plt.close()

    ### Plot raster plot - dot plot showing timing of detected peaks across ROIs (each row = ROI, each dot = peak event in time)
    # Plot raster plot of peaks maxima
    fig, raster = plot_raster(peaks_max, rois='all', nframes=6000, fs=10, title="Raster plot of peak maxima")
    plt.savefig(os.path.join(path_save, file_stem+"_raster_peak_max.png"), dpi=300)
    plt.close()

    # Plot rasterplot of peaks onsets
    fig = plot_raster(peaks_start, rois='all', nframes=6000, fs=10, title="Raster plot of peak start")
    plt.savefig(os.path.join(path_save, file_stem+"_raster_peak_start.png"), dpi=300)
    plt.close()

    # Plot population activity - sum of active components per time point (i.e., summed raster plot; each value = number of simultaneously active ROIs)
    fig=plot_population_activity (raster)
    plt.savefig(os.path.join(path_save, file_stem+"_population_activity_plot.png"), dpi=300)
    plt.close()

    #Save data
    bundle = {        
    "peaks_start": peaks_start,
    "peaks_max": peaks_max,
    "time_to_peak": time_to_peak, # in seconds
    "peaks_amp": peaks_amp,
    "raster": raster,
    "parameters":   {
        "trace filtering": {"fs":fs, "low_freq_thresh":low_freq_thresh, "high_freq_thresh":high_freq_thresh, "order":order},
        "peak finding": {"precentile_threshold": precentile_threshold, "strict": strict, "dF_thresh": dF_thresh}}
    }
    # Save the bundle
    out_path = os.path.join(path_save, f"{file_stem}_peaks.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
        
    return

   

 

In [4]:
data_source = {
"exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO":["pup2_1_spont"]
}

In [5]:
path_dist = "Data"
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        main(path_save,file_stem)
        
        print(f"Processing {file_stem} from {folder_exp}")



Processing pup2_1_spont from exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO
